# 01 – Basic DQ Checks (Bronze `orders`)

**Purpose:** a short baseline of essential checks on the three daily order files, so we know what `02_data_transformation.ipynb` has to fix. The full list of findings is in the DQ report; this notebook only re-confirms the baseline and notes where our counts differ from the report.

**Input:** `data/orders/orders_2026-10-05.csv`, `..-06.csv`, `..-07.csv` (read-only, never modified).

In [1]:
import csv, re
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/orders")
EXPECTED_COLS = ["order_id","customer_id","customer_name","email","phone","city","order_date","ship_date",
                 "status","quantity","unit_price","amount","currency","region_code","payment_mode","notes","modified_at"]
ALLOWED_STATUS = {"PLACED","SHIPPED","DELIVERED","CANCELLED","RETURNED"}
NULL_TOKENS = {"", "null", "n/a", "nan", "-", "none", "?"}

# Read with the csv module (not pandas) so malformed lines can be counted rather than crash the load
good, bad, headers = [], [], {}
for f in sorted(DATA_DIR.glob("orders_*.csv")):
    lines = f.read_text(encoding="utf-8-sig").splitlines()
    headers[f.name] = next(csv.reader([lines[0]]))
    for n, line in enumerate(lines[1:], start=2):
        r = next(csv.reader([line]))
        (good if len(r) == 17 else bad).append((f.name, n, len(r), r))
df = pd.DataFrame([r for *_, r in good], columns=EXPECTED_COLS)
print(f"files={len(headers)}  total data lines={len(good)+len(bad)}  well-formed={len(good)}  malformed={len(bad)}")

files=3  total data lines=5235  well-formed=5205  malformed=30


## Schema and structure

In [2]:
print("Header matches expected schema in every file:", all(h == EXPECTED_COLS for h in headers.values()))
print("Malformed lines by field count:", pd.Series([n for _, _, n, _ in bad]).value_counts().to_dict())
print("Malformed lines per file:", pd.Series([f for f, *_ in bad]).value_counts().to_dict())

Header matches expected schema in every file: True
Malformed lines by field count: {19: 18, 9: 12}
Malformed lines per file: {'orders_2026-10-05.csv': 10, 'orders_2026-10-06.csv': 10, 'orders_2026-10-07.csv': 10}


## Missing / placeholder values in important columns

In [3]:
def is_missing(s): return s.str.strip().str.lower().isin(NULL_TOKENS)
cols = ["order_id","customer_id","email","order_date","status","quantity","unit_price","amount"]
pd.DataFrame({"blank_or_placeholder": {c: int(is_missing(df[c]).sum()) for c in cols},
              "blank_only": {c: int((df[c].str.strip()=="").sum()) for c in cols}})

,blank_or_placeholder,blank_only
order_id,54,29
customer_id,42,42
email,74,26
order_date,12,0
status,26,14
quantity,0,0
unit_price,0,0
amount,66,22


## Duplicates

In [4]:
key_ok = df[df.order_id.str.fullmatch(r"\d+")]
print("Exact duplicate rows (all 17 columns):", int(df.duplicated().sum()))
print("Rows sharing an order_id with another row (valid ids only):", int(key_ok.order_id.duplicated(keep=False).sum()))
print("Distinct order_ids duplicated with *different* content:",
      key_ok[key_ok.order_id.duplicated(keep=False) & ~key_ok.duplicated(keep=False)].order_id.nunique())

Exact duplicate rows (all 17 columns): 75
Rows sharing an order_id with another row (valid ids only): 259
Distinct order_ids duplicated with *different* content: 58


## Unparseable types and formats

In [5]:
num = lambda s: pd.to_numeric(s.str.replace(r"[₹,]", "", regex=True), errors="coerce")
checks = {
  "order_id not integer":      (~df.order_id.str.fullmatch(r"\d+")).sum(),
  "quantity not integer":      (~df.quantity.str.fullmatch(r"-?\d+")).sum(),
  "quantity <= 0":             (num(df.quantity) <= 0).sum(),
  "amount not numeric (raw)":  pd.to_numeric(df.amount, errors="coerce").isna().sum(),
  "amount not numeric (after removing ₹ and ,)": num(df.amount).isna().sum(),
  "amount < 0":                (num(df.amount) < 0).sum(),
  "unit_price not numeric":    pd.to_numeric(df.unit_price, errors="coerce").isna().sum(),
  "order_date not yyyy-mm-dd": (~df.order_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum(),
  "ship_date not yyyy-mm-dd (non-blank)": (~df.ship_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}|")).sum(),
  "email fails basic regex":   (~df.email.str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s.]+")).sum(),
  "email fails ASCII-only regex (closest to the report's 642)": (~df.email.str.fullmatch(r"(?i)[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}")).sum(),
  "status outside allowed set": (~df.status.isin(ALLOWED_STATUS)).sum(),
}
pd.Series(checks, name="rows").astype(int).to_frame()

,rows
order_id not integer,54
quantity not integer,29
quantity <= 0,25
amount not numeric (raw),175
"amount not numeric (after removing ₹ and ,)",66
amount < 0,88
unit_price not numeric,0
order_date not yyyy-mm-dd,359
ship_date not yyyy-mm-dd (non-blank),303
email fails basic regex,254


In [6]:
d_order = pd.to_datetime(df.order_date, format="%Y-%m-%d", errors="coerce")
d_ship  = pd.to_datetime(df.ship_date,  format="%Y-%m-%d", errors="coerce")
print("ship_date < order_date (ISO-formatted rows only):", int((d_ship < d_order).sum()))
print("Distinct order_date shapes:", df.order_date.str.replace(r"\d","9",regex=True).str.replace(r"[A-Za-z]{3}","Mon",regex=True).value_counts().to_dict())

ship_date < order_date (ISO-formatted rows only): 145
Distinct order_date shapes: {'9999-99-99': 4846, '99/99/9999': 216, '9999/99/99': 102, '99-99-9999': 16, 'Mon 9 9999': 13, 'N/A': 12}


## Baseline summary vs. the DQ report

| Area | This notebook | DQ report | Note |
|---|---|---|---|
| Rows | 5,205 well-formed + 30 malformed = 5,235 | 5,235 | Matches. The report's row count counts the 30 malformed lines too. |
| Malformed CSV lines | 18 lines with 19 fields, 12 with 9 fields | not listed as a check | The report's checks ignore row structure; the notebook has to handle it before any column check. |
| order_id | 29 blank + 25 `NULL` | 29 null, 25 not bigint | Matches. |
| Exact duplicates | 75 | 75 | Matches. |
| Status / email / date / amount / quantity | see tables above | 421 / 642 / 374 / 153 / 29 | Small differences come from the 30 malformed lines. Email differs because the report's regex is ASCII-only (646 here vs 642) while a structural check gives ~250. We do not force the counts to match. |
| Negative amounts, ship < order | see tables above | 88, 145 | ship < order can only be fully counted after non-ISO dates are parsed (done in notebook 02). |

**Takeaway:** the baseline confirms the report. Issues to fix in `02_data_transformation.ipynb`: malformed lines, placeholders, mixed date formats, ₹/comma amounts, word/decimal quantities, invalid emails, unsupported statuses, exact duplicates, duplicate order_ids, missing order_id/customer_id, negative amounts and ship-before-order dates.